# 07 · Unity Catalog — gobierno de datos

Unity Catalog centraliza **permisos, linaje, descubrimiento y clasificación** de todos los activos.
En CFE permite exponer la capa **oro** a otras herramientas de forma gobernada.

En este lab: jerarquía de objetos, permisos (RBAC), `information_schema`, tags de clasificación,
linaje, máscaras de columna para datos sensibles (montos de contrato) y un **dashboard de monitoreo**
de la plataforma con system tables, generado con Genie Code.

> Requiere haber corrido el lab **01** (usa tu tabla `delta_obras`).

In [0]:
%run ./_setup

## 1. Jerarquía: catálogo › esquema › tabla
Todo objeto se referencia por su nombre de 3 niveles `catalog.schema.table`.

In [0]:
display(spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{WORK_SCHEMA}`"))

## 2. Permisos (RBAC) — GRANT / REVOKE
```sql
GRANT USE CATALOG ON CATALOG dipidev TO `analistas_construccion`;
GRANT USE SCHEMA, SELECT ON SCHEMA <tu_esquema> TO `analistas_construccion`;
GRANT SELECT ON TABLE <tu_esquema>.gold_obras_por_estado TO `analistas_construccion`;
REVOKE SELECT ON TABLE <tu_esquema>.silver_obras_cfe FROM `analistas_construccion`;
```

In [0]:
display(spark.sql(f"SHOW GRANTS ON TABLE {tbl('delta_obras')}"))

## 3. `information_schema` — metadatos consultables con SQL

In [0]:
display(spark.sql(f"""
  SELECT table_name, table_type, created, last_altered
  FROM `{CATALOG}`.information_schema.tables
  WHERE table_schema = '{WORK_SCHEMA}'
  ORDER BY created DESC
"""))

## 4. Tags de clasificación
Etiqueta objetos/columnas para gobierno (capa medallion, sensibilidad, dominio).

In [0]:
spark.sql(f"ALTER TABLE {tbl('delta_obras')} SET TAGS ('capa' = 'bronze', 'dominio' = 'construccion')")
spark.sql(f"ALTER TABLE {tbl('delta_obras')} ALTER COLUMN ImporteContratoMxn SET TAGS ('sensibilidad' = 'alta', 'tipo_dato' = 'financiero')")
spark.sql(f"ALTER TABLE {tbl('delta_obras')} ALTER COLUMN NombreObra SET TAGS ('tipo_dato' = 'descriptivo')")

display(spark.sql(f"""
  SELECT tag_name, tag_value FROM `{CATALOG}`.information_schema.table_tags
  WHERE schema_name = '{WORK_SCHEMA}' AND table_name = 'delta_obras'
"""))

## 4b. Asignar tags desde Catalog Explorer (interfaz gráfica)

Además de SQL, puedes etiquetar tablas y columnas directamente desde la UI.

### Etiquetar una tabla
1. En el menú lateral izquierdo, haz clic en **Catalog** (icono de base de datos)
2. Navega: `dipidev` → `default` → `delta_obras`
3. En la pestaña **Overview**, busca la sección **Tags**
4. Haz clic en **✚ Add tag**
5. Escribe la **clave** (ej: `capa`) y el **valor** (ej: `bronze`)
6. Haz clic en **Add** para confirmar

### Etiquetar una columna
1. En la misma tabla, ve a la pestaña **Columns** (o **Schema**)
2. Haz clic en la columna que quieras etiquetar (ej: `ImporteContratoMxn`)
3. En el panel de detalle de la columna, busca **Tags**
4. Haz clic en **✚ Add tag**
5. Escribe clave: `sensibilidad`, valor: `alta`
6. Confirma con **Add**

### ✅ Ejercicio
Desde la UI, agrega estos tags a `delta_obras`:

| Nivel | Clave | Valor |
|---|---|---|
| Tabla | `dominio` | `construccion` |
| Tabla | `responsable` | `tu_nombre` |
| Columna `ImporteContratoMxn` | `sensibilidad` | `alta` |
| Columna `ImporteContratoMxn` | `pii` | `financiero` |
| Columna `NombreObra` | `tipo_dato` | `descriptivo` |

Después, verifica tus tags con la celda de arriba o con esta consulta:
```sql
SELECT * FROM dipidev.information_schema.column_tags
WHERE table_name = 'delta_obras'
```

## 5b. Aplicar política de enmascaramiento desde Catalog Explorer

Las **column masks** ocultan datos sensibles (montos, nombres, RFC) según el grupo del usuario.
Se necesitan dos pasos: crear la función de máscara y aplicarla a la columna.

### Paso 1 — Crear la función de máscara (SQL, una sola vez)
Ejecuta la siguiente celda para crear la función. Solo los miembros del grupo `finanzas_cfe`
verán el valor real; el resto verá `NULL`.

### Paso 2 — Aplicar la máscara desde la UI
1. En **Catalog Explorer**, navega a `dipidev` → `default` → `delta_obras`
2. Ve a la pestaña **Columns** (o **Schema**)
3. Haz clic en la columna `ImporteContratoMxn`
4. En el panel de detalle, busca la sección **Column mask**
5. Haz clic en **Add column mask**
6. En el diálogo:
   * **Function catalog**: `dipidev`
   * **Function schema**: `default`
   * **Function name**: selecciona `mask_importe`
7. Haz clic en **Apply** para activar la máscara

### ¿Cómo verificar?
* Los usuarios **fuera** del grupo `finanzas_cfe` verán `NULL` en `ImporteContratoMxn`
* Los usuarios **dentro** del grupo verán el valor real
* En la UI, la columna mostrará un icono de 🔒 candado indicando que tiene máscara

### Para remover la máscara (desde la UI)
1. Haz clic en la columna `ImporteContratoMxn`
2. En **Column mask**, haz clic en **Remove**

### Para remover la máscara (SQL)
```sql
ALTER TABLE dipidev.default.delta_obras
  ALTER COLUMN ImporteContratoMxn DROP MASK;
```

In [0]:
%sql
-- Paso 1: Crear la función de máscara
-- Solo los miembros de 'finanzas_cfe' ven el importe real
CREATE OR REPLACE FUNCTION dipidev.default.mask_importe(valor DOUBLE)
  RETURN CASE
    WHEN is_account_group_member('finanzas_cfe') THEN valor
    ELSE NULL
  END

In [0]:
%sql
-- Verifica: ¿qué ve un usuario sin el grupo finanzas_cfe?
-- (Si tú estás en el grupo, verás el valor real)
SELECT IdObra, NombreObra, ImporteContratoMxn
FROM dipidev.default.delta_obras
LIMIT 5

## 5. Linaje
UC captura linaje automáticamente. Abre tu tabla `delta_obras` en **Catalog Explorer → Lineage**.

## 6. (Avanzado) Máscara de columna para datos financieros
```sql
CREATE OR REPLACE FUNCTION <tu_esquema>.mask_importe(v DOUBLE)
  RETURN CASE WHEN is_account_group_member('finanzas_cfe') THEN v ELSE -1.0 END;

ALTER TABLE <tu_esquema>.delta_obras
  ALTER COLUMN ImporteContratoMxn SET MASK <tu_esquema>.mask_importe;
```

## 7. Dashboard de monitoreo de la plataforma (system tables + Genie Code)

Unity Catalog no solo gobierna los datos: tambien **registra como se usan**. Las **system tables** (`system.*`) son tablas de solo lectura con la actividad de la cuenta: consultas, accesos, linaje, uso de Genie y mas.

En esta actividad vas a generar con **Genie Code** un dashboard de monitoreo para el equipo de plataforma de CFE.

| Pregunta de negocio | System table |
|---|---|
| Cuantas consultas se ejecutan, que tan rapido y cuantas fallan | `system.query.history` |
| Quienes son los usuarios activos y los mas intensivos | `system.query.history` |
| Que tablas se consultan mas | `system.access.table_lineage` |
| Quien accede a columnas **sensibles** (tags del paso 4) | `system.access.column_lineage` + `information_schema.column_tags` |
| Intentos de acceso denegados y cambios de permisos | `system.access.audit` |
| Uso de **Genie** (espacios, preguntas, usuarios) | `system.query.history` (`query_source.genie_space_id`) |
| Uso de **Genie Code** (el asistente) | `system.access.assistant_events` |
| Dashboards activos y sus consultas | `system.query.history` (`query_source.dashboard_id`) |

### Requisito: permisos sobre system tables
Las system tables requieren que un **administrador** otorgue acceso. Si la celda de verificacion marca `SIN ACCESO`, pide al instructor que ejecute (ajusta el grupo):

```sql
GRANT USE SCHEMA ON SCHEMA system.query  TO `participantes_workshop`;
GRANT SELECT     ON SCHEMA system.query  TO `participantes_workshop`;
GRANT USE SCHEMA ON SCHEMA system.access TO `participantes_workshop`;
GRANT SELECT     ON SCHEMA system.access TO `participantes_workshop`;
```

In [0]:
# Verifica a que system tables tienes acceso antes de pedir el dashboard
SYSTEM_TABLES = [
    "system.query.history",
    "system.access.audit",
    "system.access.table_lineage",
    "system.access.column_lineage",
    "system.access.assistant_events",
]

acceso = {}
for t in SYSTEM_TABLES:
    try:
        spark.sql(f"SELECT 1 FROM {t} LIMIT 1").collect()
        acceso[t] = True
    except Exception as e:
        acceso[t] = False

for t, ok in acceso.items():
    print(f"{'OK        ' if ok else 'SIN ACCESO'}  {t}")

if not all(acceso.values()):
    print("\nPide al instructor los GRANT de la celda anterior. Puedes generar el prompt de todos modos.")

In [0]:
from databricks.sdk import WorkspaceClient

DIAS = 30  # ventana de analisis
usuario = spark.sql("SELECT current_user()").first()[0]
nombre = usuario.split("@")[0].replace(".", " ").title()
try:
    WORKSPACE_ID = str(WorkspaceClient().get_workspace_id())
except Exception:
    WORKSPACE_ID = "<workspace_id>"

dashboard_name = f"Monitoreo Plataforma CFE - {nombre}"

prompt = f"""
Crea un dashboard de AI/BI para monitorear el uso de la plataforma Databricks de CFE.
Nombre del dashboard: {dashboard_name}
Catalogo del caso: {CATALOG} | Esquema: {WORK_SCHEMA} | Workspace ID: {WORKSPACE_ID}
Ventana de analisis: ultimos {DIAS} dias.

===== FUENTES (system tables) =====
- system.query.history             -> consultas (usuario, duracion, estado, origen: dashboard / Genie / notebook)
- system.access.table_lineage      -> lecturas y escrituras por tabla
- system.access.column_lineage     -> lecturas por columna
- system.access.audit              -> eventos de auditoria (accesos, permisos, errores)
- system.access.assistant_events   -> uso de Genie Code (columna de usuario: initiated_by)
- {CATALOG}.information_schema.column_tags -> columnas etiquetadas (sensibilidad = 'alta')

===== REGLAS =====
1. Antes de escribir cada SQL, revisa las columnas reales con DESCRIBE; no inventes columnas.
2. Filtra siempre por workspace_id = '{WORKSPACE_ID}' y por los ultimos {DIAS} dias
   (usa event_date / start_time segun la tabla, para aprovechar el particionado).
3. Usa comillas simples para literales de texto.
4. En table_lineage / column_lineage, limita al catalogo '{CATALOG}' para enfocarte en el caso CFE.
5. Convierte duraciones de milisegundos a segundos y redondea a 2 decimales.
6. Nombres de datasets y widgets en espanol. Maximo 8 widgets por pagina.
7. Agrega un filtro de fechas en cada pagina.

===== DATASETS =====
1. kpis_consultas (1 fila) desde system.query.history:
   total_consultas, usuarios_activos (distinct executed_by), duracion_promedio_seg,
   duracion_p95_seg, pct_fallidas (execution_status = 'FAILED')
2. consultas_por_dia: fecha, total_consultas, usuarios_activos, duracion_promedio_seg
3. top_usuarios: executed_by, total_consultas, duracion_total_seg, consultas_fallidas (top 15)
4. consultas_lentas: statement_id, executed_by, inicio, duracion_seg, filas_leidas, origen,
   texto (primeros 200 caracteres de statement_text) (top 20 por duracion)
5. tablas_mas_consultadas desde system.access.table_lineage:
   source_table_full_name, lecturas, usuarios_distintos, ultima_lectura (top 15)
6. accesos_datos_sensibles: une system.access.column_lineage con
   {CATALOG}.information_schema.column_tags (tag_name = 'sensibilidad', tag_value = 'alta')
   por catalogo, esquema, tabla y columna. Columnas: usuario (created_by), tabla, columna,
   accesos, ultimo_acceso
7. eventos_seguridad desde system.access.audit: fecha, usuario (user_identity.email),
   service_name, action_name, codigo_respuesta. Incluye accesos denegados (codigo 403)
   y cambios de permisos (action_name con 'Permission' o 'Grant')
8. uso_genie desde system.query.history donde query_source.genie_space_id no es nulo:
   genie_space_id, preguntas (consultas generadas), usuarios_distintos, duracion_promedio_seg
9. uso_genie_code desde system.access.assistant_events:
   fecha, interacciones, usuarios_distintos (initiated_by); y top usuarios de Genie Code
10. dashboards_activos desde system.query.history donde query_source.dashboard_id no es nulo:
   dashboard_id, consultas, usuarios_distintos, duracion_promedio_seg, ultima_ejecucion

===== ESTRUCTURA =====
Pagina 1 - Uso y Rendimiento
- Counters: Total consultas, Usuarios activos, Duracion promedio (seg), % fallidas
- Line: Consultas por dia
- Bar: Top usuarios por consultas
- Table: Consultas mas lentas

Pagina 2 - Datos y Seguridad
- Bar horizontal: Tablas mas consultadas
- Table: Accesos a datos sensibles (resaltar usuarios fuera del grupo finanzas_cfe si es posible)
- Bar: Eventos de seguridad por tipo (action_name)
- Table: Detalle de accesos denegados

Pagina 3 - Genie y Dashboards
- Counters: Preguntas a Genie, Usuarios de Genie, Interacciones con Genie Code, Dashboards activos
- Line: Interacciones de Genie Code por dia
- Bar: Espacios de Genie mas usados
- Table: Dashboards activos

===== ENTREGA =====
1. Resumen del dashboard
2. Tabla: pagina | widget | tipo_visual | dataset | pregunta_de_negocio
3. SQL de los 10 datasets, listo para pegar
4. Si alguna tabla no esta disponible o una columna no existe, dilo y propone alternativa
"""

print(prompt)

### Actividad
1. Ejecuta la celda de verificacion y luego la del prompt.
2. Copia el prompt y pegalo en **Genie Code** (`Cmd+I` / `Ctrl+I`) o crea un dashboard nuevo en **AI/BI Dashboards** y pegalo en su asistente.
3. Revisa cada SQL antes de usarlo: que filtre por workspace y fechas, y que las columnas existan.
4. Construye las 3 paginas y publica el dashboard.
5. Opcional: programa un refresh diario y suscribete (igual que en el lab 05).

### Checklist
* Los counters salen de datasets de **una sola fila**
* Todas las consultas filtran por `workspace_id` y por los ultimos 30 dias
* El dataset de datos sensibles usa los tags `sensibilidad = 'alta'` que aplicaste en el paso 4
* Las duraciones estan en segundos, no en milisegundos
* Genie (espacios de datos) y Genie Code (asistente) aparecen como metricas **separadas**

> Los datos de system tables pueden tardar unas horas en aparecer. Si un widget sale vacio, amplia la ventana de dias o revisa los filtros.